### Needs `01-ReferenceDFs.ipynb` to be executed first

Reads from:
- `gs_extractions_raw.json`
- `gs_extractions_raw_ynorm.json`
- `year_scope_df.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np

## Import Data

In [2]:
gs_extractions_raw  = pd.read_json("gs_extractions_raw.json")
gs_extractions_raw_ynorm = pd.read_json("gs_extractions_raw_ynorm.json")
year_scope_df       = pd.read_json("year_scope_df.json")

CATEGORIES = ["value", "unit"]
VARIANTS   = ["t1", "t2", "m1", "m2", "i1", "i2"]

#### Cleanup Import

In [3]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)

for col in pipeline_cols:
    gs_extractions_raw[col] = gs_extractions_raw[col].apply(
    lambda x: np.nan if x is None else x
)
    
for col in pipeline_cols:
    gs_extractions_raw_ynorm[col] = gs_extractions_raw_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [ ]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def eval_intern(source, result_basis) -> pd.DataFrame:
    
    result = result_basis.copy()
    
    for cat in CATEGORIES: # "value" then "unit" comparison
        for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
            result[f"{v}_{cat}_hit"] = source.apply(
                check_hit,                  # Apply check for every row
                args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    
    return result

def eval(source) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]])
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source)
    
    return gs_eval, in_place


def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## Straight Forward

In [8]:
gs_extractions_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   int64  
 5   scope            2208 non-null   str    
 6   page             490 non-null    str    
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_t1         497 non-null    object 
 12  value_t2         506 non-null    object 
 13  value_m1         468 non-null    object 
 14  value_m2         464 non-null    object 
 15  value_i1         490 non-null    object 
 16  value_i2         496 non-null    object 
 17  unit_t1          497 non-

In [9]:
just_eval, expended = eval(gs_extractions_raw)

print_eval(just_eval)

TypeError: eval_intern() missing 1 required positional argument: 'result_basis'

In [ ]:
ynorm, expended_ynorm = eval(gs_extractions_raw_ynorm)

print_eval(ynorm)

In [ ]:
def save_eval_comparison(eval_raw, eval_ynorm, filepath="eval_comparison.json"):
    """Speichert beide Evaluationen strukturiert zum Vergleich"""
    results = []
    
    for cat in CATEGORIES:
        for v in VARIANTS:
            col_raw = eval_raw[f"{v}_{cat}_hit"]
            col_ynorm = eval_ynorm[f"{v}_{cat}_hit"]
            
            results.append({
                "variant": v,
                "category": cat,
                "raw_true": col_raw.value_counts()[True],
                "ynorm_true": col_ynorm.value_counts()[True],
                "raw_false": col_raw.value_counts()[False],
                "ynorm_false": col_ynorm.value_counts()[False],
                "abs_improvmnt": col_ynorm.value_counts()[True] - col_raw.value_counts()[True],
                "raw_quota": round(col_raw.mean() * 100, 2),
                "ynorm_quota": round(col_ynorm.mean() * 100, 2),
                "delta_quota": round((col_ynorm.mean() - col_raw.mean()) * 100, 2)
            })
    
    df_results = pd.DataFrame(results)
    df_results.to_json(filepath, orient="records", indent=2)
    df_results.to_csv(filepath.replace(".json", ".csv"), index=False)
    
    return df_results

# Speichern und anzeigen
comparison = save_eval_comparison(just_eval, ynorm)
print(comparison)


# Detailed Degradation Analysis (Claude!)

In [ ]:
def analyze_degradation_detailed(raw_source, ynorm_source, raw_eval, ynorm_eval, variant, category):
    """Detaillierte Analyse: zeigt Gold-Standard vs. was raw/ynorm extrahiert haben"""

    hit_col = f"{variant}_{category}_hit"
    value_col = f"value_{variant}"
    unit_col = f"unit_{variant}"
    gs_col = category  # "value" or "unit"

    # Rows wo raw=True aber ynorm=False (Verschlechterung)
    degraded_mask = (raw_eval[hit_col] == True) & (ynorm_eval[hit_col] == False)
    degraded_idx = raw_source[degraded_mask].index.tolist()

    results = []
    for idx in degraded_idx:
        gs_val = raw_source.iloc[idx][gs_col]
        raw_val = raw_source.iloc[idx][value_col]
        ynorm_val = ynorm_source.iloc[idx][value_col]
        
        results.append({
            "report_name": raw_source.iloc[idx]["report_name"],
            "year": raw_source.iloc[idx]["year"],
            "scope": raw_source.iloc[idx]["scope"],
            "gs_value": gs_val,
            "raw_extracted": raw_val,
            "ynorm_extracted": ynorm_val,
            "raw_was_list": isinstance(raw_val, list),
            "ynorm_was_list": isinstance(ynorm_val, list),
        })

    return pd.DataFrame(results)


# m1 value: detaillierte Analyse
print("=" * 110)
print("M1 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "value")
print(f"Total degraded: {len(m1_value_details)}\n")
if len(m1_value_details) > 0:
    print(m1_value_details.to_string())
    m1_value_details.to_csv("m1_value_degradations.csv", index=False)
    print("\n✓ Saved to: m1_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 VALUE DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_value_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "value")
print(f"Total degraded: {len(m2_value_details)}\n")
if len(m2_value_details) > 0:
    print(m2_value_details.to_string())
    m2_value_details.to_csv("m2_value_degradations.csv", index=False)
    print("\n✓ Saved to: m2_value_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M1 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m1_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m1", "unit")
print(f"Total degraded: {len(m1_unit_details)}\n")
if len(m1_unit_details) > 0:
    print(m1_unit_details.to_string())
    m1_unit_details.to_csv("m1_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m1_unit_degradations.csv")
else:
    print("(No degradations)")

print("\n\n" + "=" * 110)
print("M2 UNIT DEGRADATIONS (raw=True ✓ → ynorm=False ✗)")
print("=" * 110)
m2_unit_details = analyze_degradation_detailed(gs_extractions_raw, gs_extractions_raw_ynorm, expended, expended_ynorm, "m2", "unit")
print(f"Total degraded: {len(m2_unit_details)}\n")
if len(m2_unit_details) > 0:
    print(m2_unit_details.to_string())
    m2_unit_details.to_csv("m2_unit_degradations.csv", index=False)
    print("\n✓ Saved to: m2_unit_degradations.csv")
else:
    print("(No degradations)")

m1 and m2 had indeed extracted values that the gold standard did not aknowledge. Therefore a miss-mapping of the "year"-value, which resolved to "NaN" led to correct retrieval. But the fact that it had found a value, which is now shown, the score went down.